# EDA (Eplorativ datanalays)

EDA står för *Exploratory Data Analysis* och innebär att utforska data för att förstå innehållet och hitta mönster.

**Syftet är att:**
- Undersöka datans struktur och innehåll.
- Hitta mönster och samband.
- Upptäcka saknade värden och dubbletter.
- Visualisera data med diagram.

**Vanliga metoder:**
- `df.head()` – visar de första raderna.
- `df.info()` – visar kolumner, datatyper och saknade värden.
- `df.describe()` – sammanfattar numeriska värden.
- `df.groupby()` – grupperar data för jämförelser.
- Histogram, stapeldiagram och boxplot – visualiserar data.

**Exempel:** Undersöka om köpbeteendet skiljer sig mellan män och kvinnor i olika städer.

**Kort sagt:** EDA hjälper oss att förstå datan innan vi drar slutsatser eller bygger modeller.

Stadsbiblioteket vill ha ett snabbt, körbart beslutsunderlag

1. Vad lånas mest? - genrer och medietyper
2. Var lånas mest? - fililer (vad menas med filialer?)
3. När lånas det? - månad/vecka?
4. Typsik utlåning- genomsnittlig lånetid och spridning
5. Topp-listor - topp-3 genrer/filialer

   -  .\ – letar i den aktuella projektmappen.
   -  .venv\Scripts\python.exe – startar Python från din virtuella miljö.
   -  --version – visar vilken Python-version du har.

## Förberedelser – Pythonprojekt med Git och `.venv`

### 1. Skapa och klona ett Git-repository
- Skapa ett repository på GitHub.
- Klona repositoryt till datorn med hjälp av GitHub-adressen.

### 2. Skapa en virtuell miljö (`.venv`)
En virtuell miljö isolerar projektet och dess bibliotek.

- Varje projekt kan ha egna bibliotek och versioner.
- Minskar risken för konflikter mellan bibliotek.
- Exempel på bibliotek: `pandas`, `numpy` och `matplotlib`.
- Gör det enklare att återskapa projektet på andra datorer.

**Aktivera miljön i PowerShell:**
```powershell
.\.venv\Scripts\Activate.ps1
```

När miljön är aktiverad syns vanligtvis `(.venv)` i terminalen.

### 3. Installera bibliotek
Kör i terminalen med `.venv` aktiverad:

```powershell
python -m pip install pandas matplotlib jupyter
```

- `python -m` säkerställer att pip körs med den Python-miljö som används.
- `python -m pip list` visar installerade paket.
- Biblioteken finns vanligtvis i `.venv/Lib/site-packages/`.

I en Jupyter-notebook kan jag även använda `%pip list`.

### 4. Skapa `requirements.txt`
Kör i terminalen:

```powershell
python -m pip freeze > requirements.txt
```

- Sparar installerade paket och deras versioner i en fil.
- Gör det enklare för andra att installera samma paket.
- Uppdatera filen när projektets beroenden förändras.

**Installera paket från filen:**
```powershell
python -m pip install -r requirements.txt
```

`-r` betyder att pip ska läsa paketlistan från filen.

### 5. Använd `.gitignore`
En `.gitignore` anger vilka filer och mappar Git ska ignorera.

Exempel:
```gitignore
.venv/
__pycache__/
*.csv
```

- `.venv/` – ladda inte upp den virtuella miljön.
- `__pycache__/` – ignorera tillfälliga Python-filer.
- `*.csv` – ignorera CSV-filer.

**Obs!** `*.csv` ignorerar alla CSV-filer i projektet. Om vissa CSV-filer ska delas behöver jag ange mer specifika sökvägar i stället.

### 6. När en kollega hämtar projektet

**Första gången:**

1. Klona repositoryt från GitHub.
2. Öppna projektmappen i VS Code.
3. Skapa en virtuell miljö:
   ```powershell
   python -m venv .venv
   ```
4. Aktivera miljön:
   ```powershell
   .\.venv\Scripts\Activate.ps1
   ```
5. Installera projektets paket:
   ```powershell
   python -m pip install -r requirements.txt
   ```

**När nya ändringar eller paket tillkommer:**

1. Hämta den senaste koden:
   ```powershell
   git pull
   ```
2. Installera eventuella nya eller uppdaterade paket:
   ```powershell
   python -m pip install -r requirements.txt
   ```

`git pull` hämtar filerna från GitHub, men installerar inte automatiskt nya Python-paket.

**Viktigt:** För att kollegan ska få dina ändringar måste du först uppdatera `requirements.txt` vid behov och pusha filen till GitHub.


In [14]:
from src.io_utils import load_data, coerce_numeric

# för att använda coerce_numeric kapslar vi in vår load data

# 1. load_data() läser in CSV-filen
# 2. coerce_numeric() konverterar utvalda kolumner till numeriska värden
# 3. Resultatet sparas i df
df = coerce_numeric(load_data("data/library_loans.csv"))

# skapar kolumnen "lone_days" (returned_date - checkout_date)
df["loan_days"] = (df["returned_date"] - df["checkout_date"]).dt.days.astype("Int64") 

# ta bort alla rader där lone_days saknar värde och sparar som en separat kopia
df = df.dropna(subset=["loan_days"]).copy() 

# filtrera fram rader där lone_days > 0 och spara som separat kopia
df = df[df["loan_days"] > 0].copy()     

# Skapar kolumnen "month" genom att omvandla checkout_date till månadsperioder
# och sedan till timestamps för månadens första dag.
df["month"] = df["checkout_date"].dt.to_period("M").dt.to_timestamp()

# skapar kolumnen "week" med veckans startdatum (måndag)
# exempel: 2025-06-18 blir 2025-06-16 (måndag)
# detta gör det enklare att gruppera och analysera utlåningen per vecka
df["week"] = df["checkout_date"].dt.to_period("W").dt.start_time



df.info()
df.head(3)



<class 'pandas.DataFrame'>
Index: 87 entries, 2 to 99
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   loan_id           87 non-null     str           
 1   checkout_date     87 non-null     datetime64[us]
 2   branch            87 non-null     str           
 3   genre             87 non-null     str           
 4   item_type         87 non-null     str           
 5   patron_age_group  87 non-null     str           
 6   patron_age        87 non-null     int64         
 7   returned_date     87 non-null     datetime64[us]
 8   overdue_days      87 non-null     float64       
 9   fine_amount       87 non-null     float64       
 10  loan_days         87 non-null     Int64         
 11  month             87 non-null     datetime64[us]
 12  week              87 non-null     datetime64[us]
dtypes: Int64(1), datetime64[us](4), float64(2), int64(1), str(5)
memory usage: 9.6 KB


,loan_id,checkout_date,branch,genre,item_type,patron_age_group,patron_age,returned_date,overdue_days,fine_amount,loan_days,month,week
2,L0003,2025-07-17,Söder,Ljudbok,E-bok,Barn,8,2025-08-04,0.0,0.0,18,2025-07-01,2025-07-14
4,L0005,2025-02-02,Öster,Barn,Tidskrift,Senior,65,2025-02-07,0.0,0.0,5,2025-02-01,2025-01-27
6,L0007,2025-04-28,Norr,Fakta,E-bok,Unga,16,2025-05-11,0.0,0.0,13,2025-04-01,2025-04-28
